# Capital Markets Agentic Capstone — run it from Colab

The fastest way to get a **public URL for a class** with no hosting account at all.
Google Colab is free, every trainee already has an account, and the tunnel below
needs no signup.

**What you get:** a URL anyone can open, running the full console — RAG, agent,
MCP multi-agent, tool console, test runner and findings.

**What to know before relying on it:**

- The URL lives as long as the Colab session. It dies when the runtime stops
  (roughly 90 minutes idle, or ~12 hours maximum), and the next run gives a new
  URL. That is fine for a class day, wrong for a permanent link.
- The tunnel URL is public and unguessable. Anyone who has it can use the app.
  There are no keys on the server, so there is nothing to steal — but stop the
  runtime when the session ends.
- Students still bring their own model key in the browser, exactly as on any
  other deployment.

Run the cells in order.

## 1. Get the project

Either upload `qt-capital-markets-agentic-capstone.zip` using the file browser on
the left, or set `GIT_URL` below to your repository. The upload path is the
default because it needs nothing set up.

In [ ]:
GIT_URL = ""  # e.g. "https://github.com/you/qtcap.git" — leave blank to upload the zip

import os, glob, subprocess, sys, zipfile

PROJECT = None

if GIT_URL:
    subprocess.run(["git", "clone", "--depth", "1", GIT_URL, "/content/qtcap_src"], check=True)
    PROJECT = "/content/qtcap_src"
else:
    zips = sorted(glob.glob("/content/*.zip"))
    if not zips:
        try:
            from google.colab import files
            print("Choose qt-capital-markets-agentic-capstone.zip …")
            uploaded = files.upload()
            zips = [f"/content/{name}" for name in uploaded]
        except ImportError:
            raise SystemExit("Not running in Colab. Upload the zip to /content or set GIT_URL.")
    with zipfile.ZipFile(zips[0]) as zf:
        zf.extractall("/content/extracted")
    # The zip contains a single top-level folder.
    inner = [p for p in glob.glob("/content/extracted/*") if os.path.isdir(p)]
    PROJECT = inner[0] if inner else "/content/extracted"

os.chdir(PROJECT)
print("Project:", PROJECT)
print("Contains:", sorted(os.listdir(PROJECT))[:12])

## 2. Install

Runtime dependencies only — about thirty seconds. No API key is required: the
default backend is a deterministic offline stub.

In [ ]:
!pip install -q -r requirements.txt
print("\ninstalled")

## 3. Sanity check before you hand out a URL

Runs a slice of the shipped suite. If this is not green, do not hand out the
link — fix it first. The full 327-case run takes about half a minute; this is a
quick subset.

In [ ]:
!python tools/run_suite.py red --category prompt_injection --quiet
!python tools/run_suite.py blue --category tool_selection --quiet

## 4. Start the app and open a public tunnel

`cloudflared` needs no account. If it fails, the next cell offers an ngrok
fallback (that one does need a free token).

In [ ]:
import os, re, subprocess, sys, time, urllib.request

PORT = 8000

# A public URL serves more than one person, so switch on the protections a
# shared instance needs: per-client rate limits, a body cap and strict redaction.
env = {**os.environ, "QTCAP_PUBLIC_MODE": "1", "QTCAP_LLM_PROVIDER": "stub",
       "QTCAP_MARKET_MODE": "auto"}

server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.api.main:app",
     "--host", "0.0.0.0", "--port", str(PORT), "--log-level", "warning"],
    env=env, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)

for _ in range(60):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/health", timeout=2).read()
        print("app is up")
        break
    except Exception:
        if server.poll() is not None:
            raise SystemExit(f"server died:\n{server.stderr.read()[-1500:]}")
        time.sleep(1)
else:
    raise SystemExit("server did not become healthy")

# cloudflared quick tunnel — no account needed
!wget -q -O /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /tmp/cloudflared

tunnel = subprocess.Popen(["/tmp/cloudflared", "tunnel", "--url", f"http://localhost:{PORT}",
                           "--no-autoupdate"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

public_url = None
deadline = time.time() + 90
while time.time() < deadline:
    line = tunnel.stdout.readline()
    if not line:
        if tunnel.poll() is not None:
            break
        continue
    found = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if found:
        public_url = found.group(0)
        break

if public_url:
    print("\n" + "=" * 68)
    print("  Share this with your batch:")
    print("  " + public_url)
    print("=" * 68)
    print("\n  It stays up while this Colab runtime is running.")
    print("  Tell students: open 'Model & key' and paste a free Groq key.")
else:
    print("cloudflared did not produce a URL — run the ngrok fallback cell below.")

### Fallback: ngrok

Only if cloudflared failed. Needs a free token from
<https://dashboard.ngrok.com/get-started/your-authtoken>.

In [ ]:
NGROK_TOKEN = ""  # paste your free token here

if NGROK_TOKEN:
    !pip install -q pyngrok
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_TOKEN)
    print("Share this with your batch:", ngrok.connect(PORT).public_url)
else:
    print("Set NGROK_TOKEN above first.")

## 5. While the class runs

Keep this notebook tab open — closing it stops the runtime and kills the URL.
The cell below shows the findings your students have filed, so you can watch
them come in during the session.

In [ ]:
import json, urllib.request

with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/issues?limit=50") as r:
    data = json.load(r)

stats = data["stats"]
print(f"{stats['total']} findings · {stats['open']} open · {stats['critical_open']} critical open\n")
for issue in data["issues"][:25]:
    print(f"  {issue['id']}  {issue['severity']:8}  {issue['reporter']:12}  {issue['title'][:60]}")

## 6. Export the findings before you stop

The runtime's disk is wiped when the session ends, so download the findings
while the class is still live. This is the artefact worth keeping.

In [ ]:
import urllib.request

for fmt, name in (("markdown", "findings.md"), ("csv", "findings.csv"), ("json", "findings.json")):
    with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/issues/export?fmt={fmt}") as r:
        open(f"/content/{name}", "wb").write(r.read())
    print("wrote /content/" + name)

try:
    from google.colab import files
    files.download("/content/findings.md")
except Exception as exc:
    print("Download them from the file browser on the left.", exc)